# OmniPath table ingestion

`from_omnipath` can consume an OmniPath-style interaction table
directly. This notebook uses a local table so it is deterministic.


In [ ]:
import annnet as an

an.info()


## Build from a local OmniPath-style table


In [ ]:
import polars as pl

interactions = pl.DataFrame(
    {
        'source': ['EGF', 'EGFR', 'EGFR', 'EGFR', 'RAS', 'MEK'],
        'target': ['EGFR', 'RAS', 'RAS', 'GRB2', 'MEK', 'ERK'],
        'interaction_id': [
            'EGF_EGFR',
            'EGFR_RAS_primary',
            'EGFR_RAS_secondary',
            'EGFR_GRB2_complex',
            'RAS_MEK',
            'MEK_ERK',
        ],
        'is_directed': [True, True, True, False, True, True],
        'curation_score': [0.95, 0.88, 0.63, 0.76, 0.82, 0.79],
        'consensus_direction': [1, 1, 1, 0, 1, 1],
        'source_database': [
            'omnipath',
            'omnipath',
            'literature',
            'complexportal',
            'pathwayextra',
            'kinaseextra',
        ],
    }
)

G = an.from_omnipath(
    interactions,
    source_col='source',
    target_col='target',
    edge_id_col='interaction_id',
    directed_col='is_directed',
    weight_col='curation_score',
    edge_attr_cols=['consensus_direction', 'source_database'],
    load_vertex_annotations=False,
)

print('shape:', G.shape)
G.views.edges().select(
    ['edge_id', 'source', 'target', 'effective_weight', 'source_database']
)


## Add analysis context as slices


In [ ]:
rows = list(G.views.edges().iter_rows(named=True))
edge_label = {
    row['edge_id']: f"{row['source']} -> {row['target']}"
    for row in rows
}
high_confidence = [
    row['edge_id']
    for row in rows
    if row['effective_weight'] >= 0.85
]
G.slices.add('high_confidence')
G.slices.add_edges('high_confidence', high_confidence)

print(
    'high-confidence interactions:',
    [edge_label[eid] for eid in sorted(G.slices.edges('high_confidence'))],
)


## Draw the prior network


In [ ]:
from annnet.utils import plotting

plotting.plot(
    G,
    backend='graphviz',
    show_edge_labels=True,
    edge_label_keys=['source_database'],
)


AnnNet can use OmniPath-style tables as prior knowledge while
keeping confidence, provenance, and downstream contexts in one
graph object.
